In [ ]:
# Import packages
import os
from pathlib import Path
import anndata as ad

os.chdir(Path.cwd().parent)
project_dir = Path("/home/mcaskey/10XvParse/")

import matplotlib.pyplot as plt
from XvP_utils import plotting

In [2]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

# Preprocess

In [3]:
kb_H1_dir = project_dir / "Data/Analysis_3/parse_H1/kb_python/all_out"
kb_H2_dir = project_dir / "Data/Analysis_3/parse_H2/kb_python/all_out"
polyT_bcs_file = project_dir / "Configs/Analysis_3/parse_H1/r1_T.txt"
rando_bcs_file = project_dir / "Configs/Analysis_3/parse_H1/r1_R.txt"
replacement_bcs_file = project_dir / "Configs/Analysis_3/parse_H1/replace.txt"
bc_to_wells_file = project_dir / "Configs/Analysis_3/parse_H1/bcs_to_wells.txt"

In [ ]:
# load data
H1_data = plotting.init_processing('parse_H1', kb_dir=kb_H1_dir, data_title = "Parse H1", modified=True)
H2_data = plotting.init_processing('parse_H2', kb_dir=kb_H2_dir, data_title = "Parse H2", modified=True)
unmodified_H1_data = plotting.init_processing('parse_H1', kb_dir=kb_H1_dir, data_title = "Parse H1", modified=False)
unmodified_H2_data = plotting.init_processing('parse_H2', kb_dir=kb_H2_dir, data_title = "Parse H2", modified=False)
parse_data = ad.concat([H1_data, H2_data], join='outer')
unmodified_data = ad.concat([unmodified_H1_data, unmodified_H2_data], join = 'outer')

In [12]:
parse_data.uns['title'] = 'Parse'
unmodified_data.uns['title'] = 'Parse'

In [13]:
# count in polyT and randO barcodes and extract the corresponding data from the unmodified data
with open(polyT_bcs_file, 'r') as f:
    polyT_bcs = [line.strip() for line in f]
with open(rando_bcs_file, 'r') as f:
    rando_bcs = [line.strip() for line in f]

polyT_data = unmodified_data[unmodified_data.obs_names.str.endswith(tuple(polyT_bcs))]
randO_data = unmodified_data[unmodified_data.obs_names.str.endswith(tuple(rando_bcs))]

# replace randO barcodes with polyT barcodes in randO data
with open(replacement_bcs_file, 'r') as f:
    for line in f:
        rando_bc, polyT_bc = line.strip().split()
        randO_data.obs_names = randO_data.obs_names.str.replace(f"{rando_bc}$", polyT_bc[1:], regex=True)

# Add well information to the data
bc_to_wells = {}
with open(bc_to_wells_file, 'r') as f:
    for line in f:
        bc, well = line.strip().split()
        bc_to_wells[bc] = well

randO_data.obs['well'] = randO_data.obs_names.str[-8:].map(bc_to_wells)
polyT_data.obs['well'] = polyT_data.obs_names.str[-8:].map(bc_to_wells)
parse_data.obs['well'] = parse_data.obs_names.str[-8:].map(bc_to_wells)

/tmp/ipykernel_663896/2992652321.py:24: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  polyT_data.obs['well'] = polyT_data.obs_names.str[-8:].map(bc_to_wells)


In [14]:
# Add polyT and randO count counts to parse data
parse_data.obs['polyT_counts'] = 0
parse_data.obs.loc[polyT_data.obs_names, 'polyT_counts'] = polyT_data.obs['n_counts']

parse_data.obs['randO_counts'] = 0
parse_data.obs.loc[randO_data.obs_names, 'randO_counts'] = randO_data.obs['n_counts']

# Parse Well Analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

plotting.violin_by_well(ax[0], parse_data.obs)
plotting.violin_by_cell(ax[1], parse_data.obs)

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

plotting.plot_by_well(ax[0], parse_data.obs)
plotting.plot_by_cell(ax[1], parse_data.obs)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

parse_filtered = plotting.knee_plot(ax, parse_data, cutoff = 500)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

plotting.violin_by_well(ax[0], parse_filtered.obs)
plotting.violin_by_cell(ax[1], parse_filtered.obs)

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

plotting.plot_by_well(ax[0], parse_filtered.obs)
plotting.plot_by_cell(ax[1], parse_filtered.obs)

plt.tight_layout()
plt.show()